# 01 — Trading near delivery

Materialises the **shared dimensions** and the bronze → silver → gold tables described in [`../specifications/01-trading-near-delivery.md`](../specifications/01-trading-near-delivery.md).

**Shared dimensions (owned here, used by 02 & 03)**
- `short_term_dim_zones`
- `short_term_dim_assets`
- `short_term_dim_intervals`

**Capability tables**
- `short_term_bronze_weather_obs`, `short_term_bronze_scada_telemetry`
- `short_term_silver_generation_forecast`, `short_term_silver_nominations`
- `short_term_gold_squaring_actions` (residual delta + recommended squaring per interval)
- `short_term_gold_imbalance_exposure` (net imbalance + projected cash-out)
- `short_term_gold_near_delivery_summary` (headline KPIs per day)
- `short_term_gold_backtest_runs` (strategy variants — Delta time-travel + MLflow story)

**Note:** `main.md` frames this as sub-second streaming (Real-Time Mode + DLT). For a reproducible demo this notebook **materialises the same gold in batch** over a fixed set of 15-minute intervals, carrying `forecast_ts` / `snapshot_ts` so the app can show "as last updated".

**UC comments:** [`uc_table_comments.py`](./uc_table_comments.py) — applied in the final cell.

In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## Short-term trading — common preamble
# MAGIC Reads catalog/schema from widgets or environment variables (schema created by 00_reset_demo_schema).

# COMMAND ----------

import os
import math
import random
import datetime as dt

from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"

print(f"Target: {CATALOG}.{SCHEMA}")

spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")

def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"

random.seed(7)

# Demo trading days: 2 history days + today. 96 x 15-minute intervals per day.
TODAY = dt.date.today()
DELIVERY_DATES = [TODAY - dt.timedelta(days=2), TODAY - dt.timedelta(days=1), TODAY]
LATEST_DATE = DELIVERY_DATES[-1]
N_INTERVALS = 96
NOW_INDEX = 56  # ~14:00 "now" cursor on the latest day; earlier = settled

ZONES = ["DE", "NL", "FR", "BE", "AT"]

def interval_ts(day: dt.date, idx: int) -> dt.datetime:
    return dt.datetime.combine(day, dt.time(0, 0)) + dt.timedelta(minutes=15 * idx)

def is_settled(day: dt.date, idx: int) -> bool:
    if day < LATEST_DATE:
        return True
    if day == LATEST_DATE:
        return idx < NOW_INDEX
    return False

In [ ]:
# MAGIC %md
# MAGIC ## 1. Shared dimensions — zones, assets, intervals

# COMMAND ----------

zones = [
    Row(zone_code="DE", country="Germany",     tso_name="50Hertz/Amprion/TenneT/TransnetBW", imbalance_area="DE-LU",  eic_code="10Y1001A1001A82H", currency="EUR"),
    Row(zone_code="NL", country="Netherlands", tso_name="TenneT NL",                          imbalance_area="NL",     eic_code="10YNL----------L", currency="EUR"),
    Row(zone_code="FR", country="France",      tso_name="RTE",                                imbalance_area="FR",     eic_code="10YFR-RTE------C", currency="EUR"),
    Row(zone_code="BE", country="Belgium",     tso_name="Elia",                               imbalance_area="BE",     eic_code="10YBE----------2", currency="EUR"),
    Row(zone_code="AT", country="Austria",     tso_name="APG",                                imbalance_area="AT",     eic_code="10YAT-APG------L", currency="EUR"),
]
(spark.createDataFrame(zones)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_dim_zones")))

# The 1,000 MW Central-Europe portfolio from main.md (DE-centric physical book).
assets = [
    Row(asset_id="CCGT_DE_001",  asset_type="CCGT",          nameplate_mw=500.0, zone_code="DE", efficiency=0.55, emission_factor_tco2_mwh=0.364, energy_capacity_mwh=None,  is_dispatchable=True,  annotation="flexible CCGT, fast start-stop"),
    Row(asset_id="WIND_DE_001",  asset_type="WIND_ONSHORE",  nameplate_mw=300.0, zone_code="DE", efficiency=None, emission_factor_tco2_mwh=None,  energy_capacity_mwh=None,  is_dispatchable=False, annotation="onshore wind fleet"),
    Row(asset_id="SOLAR_DE_001", asset_type="SOLAR_PV",      nameplate_mw=200.0, zone_code="DE", efficiency=None, emission_factor_tco2_mwh=None,  energy_capacity_mwh=None,  is_dispatchable=False, annotation="utility-scale solar PV"),
    Row(asset_id="BATT_DE_001",  asset_type="BATTERY",       nameplate_mw=100.0, zone_code="DE", efficiency=None, emission_factor_tco2_mwh=None,  energy_capacity_mwh=200.0, is_dispatchable=True,  annotation="2h duration grid battery"),
]
(spark.createDataFrame(assets)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_dim_assets")))

# 15-minute time spine across the demo trading days.
intervals = []
for day in DELIVERY_DATES:
    for idx in range(N_INTERVALS):
        start = interval_ts(day, idx)
        end = start + dt.timedelta(minutes=15)
        hour = start.hour
        intervals.append(Row(
            delivery_date=day,
            interval_start=start,
            interval_end=end,
            interval_index=idx + 1,
            hour=hour,
            is_peak=(8 <= hour < 20),
            solar_window=(10 <= hour < 16),
        ))
(spark.createDataFrame(intervals)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_dim_intervals")))

print("zones:", spark.table(fq("short_term_dim_zones")).count())
print("assets:", spark.table(fq("short_term_dim_assets")).count())
print("intervals:", spark.table(fq("short_term_dim_intervals")).count())

In [ ]:
# MAGIC %md
# MAGIC ## 2. Bronze + Silver — weather, telemetry, re-forecast, nominations

# COMMAND ----------

# Per-zone renewable fleet capacities (MW). DE matches the dim_assets portfolio.
ZONE_RENEW = {
    "DE": {"wind": 300.0, "solar": 200.0},
    "NL": {"wind": 220.0, "solar": 160.0},
    "FR": {"wind": 180.0, "solar": 120.0},
    "BE": {"wind": 140.0, "solar": 110.0},
    "AT": {"wind": 120.0, "solar": 90.0},
}

def solar_factor(idx: int) -> float:
    hour = idx * 15.0 / 60.0
    if hour <= 6 or hour >= 19:
        return 0.0
    return max(0.0, math.sin(math.pi * (hour - 6.0) / 13.0))

# Stable per-(day,zone) wind level so a day has character; gentle intra-day drift.
_wind_level = {}
for day in DELIVERY_DATES:
    for z in ZONES:
        _wind_level[(day, z)] = random.uniform(0.25, 0.70)

def wind_factor(day: dt.date, zone: str, idx: int) -> float:
    base = _wind_level[(day, zone)]
    drift = 0.10 * math.sin(idx / 96.0 * 2 * math.pi + hash(zone) % 7)
    return min(0.95, max(0.02, base + drift + random.gauss(0.0, 0.03)))

# ---- Bronze: weather observations (latest forecast_ts per interval) ----
weather_rows = []
for day in DELIVERY_DATES:
    fts = dt.datetime.combine(day, dt.time(6, 0))
    for z in ZONES:
        for idx in range(N_INTERVALS):
            wf = wind_factor(day, z, idx)
            sf = solar_factor(idx)
            weather_rows.append(Row(
                ingestion_ts=fts,
                forecast_ts=fts,
                source=random.choice(["ECMWF", "GFS", "VENDOR_X", "SATELLITE"]),
                zone_code=z,
                interval_start=interval_ts(day, idx),
                wind_speed_ms=round(3.0 + wf * 18.0, 2),
                solar_irradiance_wm2=round(sf * 950.0, 1),
                temperature_c=round(8.0 + 8.0 * math.sin(idx / 96.0 * 2 * math.pi) + random.gauss(0, 1), 1),
                cloud_cover_pct=round(max(0.0, min(100.0, (1 - sf) * 60 + random.gauss(0, 12))), 1),
            ))
(spark.createDataFrame(weather_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_bronze_weather_obs")))

# ---- Bronze: SCADA telemetry for the DE portfolio assets ----
scada_rows = []
asset_rows = spark.table(fq("short_term_dim_assets")).collect()
for day in DELIVERY_DATES:
    for idx in range(N_INTERVALS):
        if not is_settled(day, idx):
            continue
        tts = interval_ts(day, idx)
        for a in asset_rows:
            if a.asset_type == "WIND_ONSHORE":
                out = a.nameplate_mw * wind_factor(day, a.zone_code, idx)
            elif a.asset_type == "SOLAR_PV":
                out = a.nameplate_mw * solar_factor(idx)
            elif a.asset_type == "CCGT":
                out = a.nameplate_mw * (0.6 if (8 <= (idx * 15 // 60) < 20) else 0.25)
            else:  # BATTERY net output ~0 average
                out = a.nameplate_mw * random.uniform(-0.4, 0.4)
            status = "RUNNING"
            if random.random() < 0.01:
                status, out = "OUTAGE", 0.0
            scada_rows.append(Row(
                ingestion_ts=tts,
                telemetry_ts=tts,
                asset_id=a.asset_id,
                actual_output_mw=round(out, 2),
                availability_pct=round(100.0 if status == "RUNNING" else 0.0, 1),
                status=status,
            ))
(spark.createDataFrame(scada_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_bronze_scada_telemetry")))

# ---- Silver: per-asset re-forecast vs actual (renewables that deviate) ----
gen_rows = []
renew_assets = [a for a in asset_rows if a.asset_type in ("WIND_ONSHORE", "SOLAR_PV")]
for day in DELIVERY_DATES:
    fts = dt.datetime.combine(day, dt.time(6, 0))
    for idx in range(N_INTERVALS):
        for a in renew_assets:
            if a.asset_type == "WIND_ONSHORE":
                factor = wind_factor(day, a.zone_code, idx)
            else:
                factor = solar_factor(idx)
            forecast = a.nameplate_mw * factor
            # prior re-forecast slightly different (weather updated)
            prev = forecast * (1 + random.gauss(0.0, 0.06))
            actual = forecast * (1 + random.gauss(0.0, 0.04)) if is_settled(day, idx) else None
            dev = forecast - prev
            sev = "SEVERE" if abs(dev) > 0.10 * a.nameplate_mw else ("WATCH" if abs(dev) > 0.04 * a.nameplate_mw else "OK")
            gen_rows.append(Row(
                delivery_date=day,
                interval_start=interval_ts(day, idx),
                forecast_ts=fts,
                asset_id=a.asset_id,
                zone_code=a.zone_code,
                forecast_mw=round(forecast, 2),
                prev_forecast_mw=round(prev, 2),
                actual_mw=round(actual, 2) if actual is not None else None,
                forecast_deviation_mw=round(dev, 2),
                deviation_severity=sev,
            ))
(spark.createDataFrame(gen_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_silver_generation_forecast")))

# ---- Silver: commercial nominations per zone per interval ----
nom_rows = []
for day in DELIVERY_DATES:
    for z in ZONES:
        cap = ZONE_RENEW[z]
        for idx in range(N_INTERVALS):
            # nomination reflects the schedule-time expected net (hedge handover)
            expected_gen = cap["wind"] * _wind_level[(day, z)] + cap["solar"] * solar_factor(idx)
            hedge = expected_gen * 0.95
            nom_rows.append(Row(
                delivery_date=day,
                interval_start=interval_ts(day, idx),
                zone_code=z,
                nominated_mw=round(expected_gen, 2),
                hedge_mw=round(hedge, 2),
                source="HEDGE_HANDOVER",
            ))
(spark.createDataFrame(nom_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_silver_nominations")))

print("weather:", spark.table(fq("short_term_bronze_weather_obs")).count())
print("scada:", spark.table(fq("short_term_bronze_scada_telemetry")).count())
print("gen_forecast:", spark.table(fq("short_term_silver_generation_forecast")).count())
print("nominations:", spark.table(fq("short_term_silver_nominations")).count())

In [ ]:
# MAGIC %md
# MAGIC ## 3. Gold — squaring actions, imbalance exposure, near-delivery summary

# COMMAND ----------

NOW_TS = interval_ts(LATEST_DATE, NOW_INDEX)
SQUARE_THRESHOLD_MW = 5.0

def reforecast_shift(day: dt.date, zone: str, idx: int) -> float:
    # Deterministic re-forecast change vs schedule, bounded ~[-0.15, 0.15].
    h = (hash(zone) % 13)
    return 0.10 * math.sin((idx + h) / 96.0 * 2 * math.pi) + 0.05 * math.cos(idx / 12.0)

def gate_state(day: dt.date, gate_close: dt.datetime):
    mins = (gate_close - NOW_TS).total_seconds() / 60.0
    if day < LATEST_DATE or mins < 0:
        return "CLOSED", int(mins)
    if mins <= 15:
        return "CLOSING", int(mins)
    return "OPEN", int(mins)

noms = spark.table(fq("short_term_silver_nominations")).collect()

squaring_rows, imbalance_rows = [], []
for n in noms:
    day, z = n.delivery_date, n.zone_code
    idx = int(round((n.interval_start - dt.datetime.combine(day, dt.time(0, 0))).total_seconds() / 900.0))
    nominated = n.nominated_mw
    shift = reforecast_shift(day, z, idx)
    forecast_gen = nominated * (1.0 + shift)
    net_delta = forecast_gen - nominated
    if net_delta > SQUARE_THRESHOLD_MW:
        side = "SELL"
    elif net_delta < -SQUARE_THRESHOLD_MW:
        side = "BUY"
    else:
        side = "FLAT"
    is_solar = 10 <= idx * 15 // 60 < 16
    is_peak = 8 <= idx * 15 // 60 < 20
    price = 65.0 + (15.0 if is_peak else -5.0) + (-45.0 if is_solar else 0.0) + 18.0 * math.sin(idx / 96.0 * 2 * math.pi)
    gate_close = n.interval_start - dt.timedelta(minutes=30)
    gstatus, mins_to_gate = gate_state(day, gate_close)
    squaring_rows.append(Row(
        delivery_date=day,
        interval_start=n.interval_start,
        interval_index=idx + 1,
        zone_code=z,
        forecast_ts=dt.datetime.combine(day, dt.time(6, 0)),
        forecast_gen_mw=round(forecast_gen, 2),
        nominated_mw=round(nominated, 2),
        net_delta_mw=round(net_delta, 2),
        recommended_side=side,
        recommended_mw=round(abs(net_delta), 2) if side != "FLAT" else 0.0,
        expected_price_eur_mwh=round(price, 2),
        gate_close_ts=gate_close,
        minutes_to_gate=mins_to_gate,
        gate_status=gstatus,
    ))

    # Imbalance: metered vs nominated for settled intervals
    settled = is_settled(day, idx)
    metered = forecast_gen * (1.0 + random.gauss(0.0, 0.03)) if settled else None
    net_imb = (metered - nominated) if settled else None
    sys_dir = "SHORT" if math.sin((idx + hash(z) % 5) / 96.0 * 2 * math.pi) < 0 else "LONG"
    imb_price = 120.0 if sys_dir == "SHORT" else 35.0
    cashout = (-net_imb * imb_price) if net_imb is not None else None
    if net_imb is None:
        sev = "OK"
    elif abs(net_imb) > 0.10 * max(nominated, 1.0):
        sev = "BREACH"
    elif abs(net_imb) > 0.04 * max(nominated, 1.0):
        sev = "WATCH"
    else:
        sev = "OK"
    imbalance_rows.append(Row(
        delivery_date=day,
        interval_start=n.interval_start,
        zone_code=z,
        nominated_mw=round(nominated, 2),
        metered_mw=round(metered, 2) if metered is not None else None,
        net_imbalance_mw=round(net_imb, 2) if net_imb is not None else None,
        system_balance_direction=sys_dir,
        imbalance_price_eur_mwh=round(imb_price, 2),
        projected_cashout_eur=round(cashout, 2) if cashout is not None else None,
        exposure_severity=sev,
    ))

(spark.createDataFrame(squaring_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_squaring_actions")))
(spark.createDataFrame(imbalance_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_imbalance_exposure")))

# ---- Near-delivery summary per day ----
sq = spark.table(fq("short_term_gold_squaring_actions"))
imb = spark.table(fq("short_term_gold_imbalance_exposure"))
gen = spark.table(fq("short_term_silver_generation_forecast"))

sev_by_day = (gen.filter(F.col("deviation_severity") == "SEVERE")
    .groupBy("delivery_date").agg(F.countDistinct("asset_id").alias("n_severe_deviations")))

summary = (sq.groupBy("delivery_date").agg(
        F.sum(F.when(F.col("gate_status").isin("OPEN", "CLOSING"), F.col("net_delta_mw")).otherwise(F.lit(0.0))).alias("net_open_position_mw"),
        F.sum(F.when(F.col("gate_status") == "OPEN", 1).otherwise(0)).alias("n_intervals_open"),
        F.sum(F.when(F.col("gate_status") == "CLOSING", 1).otherwise(0)).alias("n_intervals_closing"),
        F.min(F.when(F.col("gate_status").isin("OPEN", "CLOSING"), F.col("gate_close_ts"))).alias("next_gate_close_ts"),
    )
    .join(sev_by_day, "delivery_date", "left")
    .join(
        imb.groupBy("delivery_date").agg(F.sum(F.coalesce(F.col("projected_cashout_eur"), F.lit(0.0))).alias("projected_cashout_eur")),
        "delivery_date", "left")
    .withColumn("snapshot_ts", F.to_timestamp(F.concat(F.col("delivery_date").cast("string"), F.lit(" 14:00:00"))))
    .withColumn("system_balance_direction", F.lit("SHORT"))
    .fillna(0, subset=["n_severe_deviations", "projected_cashout_eur"])
    .withColumn("headline",
        F.when(F.col("n_intervals_closing") > 0, F.lit("Intervals closing — square residual before the gate"))
         .when(F.abs(F.col("net_open_position_mw")) > 50, F.lit("Large open position — act before delivery"))
         .otherwise(F.lit("Book broadly squared for the session")))
    .select("delivery_date", "snapshot_ts", "headline", "net_open_position_mw",
            "n_intervals_open", "n_intervals_closing", "n_severe_deviations",
            "projected_cashout_eur", "system_balance_direction", "next_gate_close_ts"))

(summary.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_near_delivery_summary")))

print("squaring:", sq.count(), "imbalance:", imb.count())
display(spark.table(fq("short_term_gold_near_delivery_summary")).orderBy(F.col("delivery_date").desc()))

In [ ]:
# MAGIC %md
# MAGIC ## 4. Gold — backtested squaring strategies (Delta time-travel + MLflow story)
# MAGIC Replays the demo's historical intervals under four strategy variants. In production the inputs
# MAGIC come from `TIMESTAMP AS OF` snapshots and each variant is logged as an MLflow run.

# COMMAND ----------

hist_days = [d for d in DELIVERY_DATES if d < LATEST_DATE]
n_intervals_hist = spark.table(fq("short_term_gold_squaring_actions")) \
    .filter(F.col("delivery_date").isin(hist_days)).count()

# Illustrative realised outcomes per strategy vs a no-trade baseline.
strategies = [
    ("BASELINE_NO_TRADE",   0.0,        0.0,      0.0,   0.0,  False),
    ("WAIT_FOR_GATE",       28_500.0,   41_000.0, 61.0,  6.5,  False),
    ("THRESHOLD_DEVIATION", 47_200.0,   73_400.0, 72.0,  19.0, True),
    ("EARLY_SQUARE",        39_800.0,   95_600.0, 68.0,  41.0, False),
]
bt_rows = []
for i, (name, pnl, cashout_avoided, hit, lead, rec) in enumerate(strategies):
    bt_rows.append(Row(
        run_id=f"st01-bt-{LATEST_DATE.isoformat()}-{i:02d}",
        strategy=name,
        backtest_start_date=hist_days[0] if hist_days else LATEST_DATE,
        backtest_end_date=hist_days[-1] if hist_days else LATEST_DATE,
        n_intervals=int(n_intervals_hist),
        realized_pnl_eur=round(pnl, 2),
        cashout_avoided_eur=round(cashout_avoided, 2),
        hit_rate_pct=round(hit, 1),
        avg_minutes_before_gate=round(lead, 1),
        is_recommended=rec,
    ))
(spark.createDataFrame(bt_rows)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(fq("short_term_gold_backtest_runs")))

display(spark.table(fq("short_term_gold_backtest_runs")).orderBy(F.col("realized_pnl_eur").desc()))

In [ ]:
# MAGIC %md
# MAGIC ## 5. Row counts + Unity Catalog comments

# COMMAND ----------

for t in [
    "short_term_dim_zones",
    "short_term_dim_assets",
    "short_term_dim_intervals",
    "short_term_bronze_weather_obs",
    "short_term_bronze_scada_telemetry",
    "short_term_silver_generation_forecast",
    "short_term_silver_nominations",
    "short_term_gold_squaring_actions",
    "short_term_gold_imbalance_exposure",
    "short_term_gold_near_delivery_summary",
    "short_term_gold_backtest_runs",
]:
    print(f"  {t:42s}  {spark.table(fq(t)).count():>10,} rows")

# COMMAND ----------

from pathlib import Path

_uc_paths = []
try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _uc_paths.append(Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_paths.append(Path.cwd() / "uc_table_comments.py")

_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found next to this notebook.")

exec(_uc_py.read_text(), globals())
apply_short_term_notebook_01_comments(spark, CATALOG, SCHEMA)
print("UC comments applied for notebook 01.")